# 07 - Decision Trees, Random Forests & Gradient Boosting

Notebook 01 used `RandomForestClassifier` as a black box. This notebook opens it.

Tree models are the workhorse of tabular machine learning. On spreadsheet-shaped data
a well-tuned gradient boosting model beats a neural network most of the time.

By the end you will be able to:
1. Explain how a tree picks a split.
2. Read `feature_importances_` - and know its trap.
3. Explain **bagging** (Random Forest) vs **boosting** (Gradient Boosting).
4. Choose between them.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
np.random.seed(42)

heart = pd.read_csv("../data/heart-disease.csv")
X = heart.drop("target", axis=1)
y = heart["target"]
print(X.shape, y.value_counts().to_dict())

## 1. A single decision tree

A tree asks yes/no questions. At each node it tries every feature and every threshold,
and keeps the split that makes the two resulting groups **purest** - most one-sided in
their class labels. Purity is measured by *gini impurity* or *entropy*.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(f"train {tree.score(X_train, y_train):.3f} | test {tree.score(X_test, y_test):.3f}")

plt.figure(figsize=(20, 8))
plot_tree(tree, feature_names=X.columns, class_names=["no disease", "disease"],
          filled=True, rounded=True, fontsize=9)
plt.title("A depth-3 decision tree - readable by a human, which is the point")
plt.show()

**Read one box.** `gini` is the impurity (0 = perfectly pure), `samples` is how many
training rows reached this node, `value` is the class breakdown. The tree stops splitting
when it hits `max_depth`, or a node is pure, or too few samples remain.

This interpretability is the tree's superpower. You can print it and hand it to a doctor.

### The weakness: instability

In [ ]:
# Change the training data slightly and the tree can restructure completely.
for seed in [0, 1, 2]:
    Xs, _, ys, _ = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    t = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xs, ys)
    root = X.columns[t.tree_.feature[0]]
    print(f"seed {seed}: root split on '{root}' at {t.tree_.threshold[0]:.2f}")

A model whose top-level question changes with the sample is **high variance**. The fix is
to build many trees and average them - which is exactly what the next two models do.

## 2. Random Forest = bagging

**Bagging** (bootstrap aggregating):
1. Draw many random samples of the rows, with replacement.
2. Fit a deep tree on each one.
3. At every split, consider only a random subset of the features.
4. Average the votes.

Each tree overfits in its own direction; averaging cancels the noise out.
Trees are built **independently and in parallel**.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

for n in [1, 5, 25, 100, 300]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42)
    s = cross_val_score(rf, X, y, cv=5)
    print(f"n_estimators={n:4d} | cv accuracy {s.mean():.3f} +/- {s.std():.3f}")

Notice that the score climbs and then plateaus. **More trees never hurt a Random Forest** -
they only cost time. That is unusual and worth remembering; you cannot overfit by adding
trees here (you can with boosting).

### Key hyperparameters

| Parameter | Effect |
|---|---|
| `n_estimators` | more trees, more stable, slower. 100-500 is normal |
| `max_depth` | `None` lets each tree grow fully - usually fine for a forest |
| `max_features` | features considered per split. `"sqrt"` is the classification default |
| `min_samples_leaf` | raise it to smooth the model |
| `class_weight` | `"balanced"` for imbalanced classes |

## 3. Gradient Boosting = boosting

**Boosting** is the opposite idea. Trees are built **sequentially**, and each new tree is
trained on the *errors of the ensemble so far*. Every tree is deliberately shallow (a
"weak learner"); the sum of hundreds of them is strong.

`HistGradientBoostingClassifier` is scikit-learn's fast modern implementation. It also
handles missing values natively, with no imputer.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)
s = cross_val_score(hgb, X, y, cv=5)
print(f"HistGradientBoosting | cv accuracy {s.mean():.3f} +/- {s.std():.3f}")

In [ ]:
# learning_rate x n_iter is the central trade-off in boosting.
for lr in [0.01, 0.1, 0.5]:
    for n in [50, 200]:
        m = HistGradientBoostingClassifier(learning_rate=lr, max_iter=n, random_state=42)
        s = cross_val_score(m, X, y, cv=5)
        print(f"learning_rate={lr:<5} max_iter={n:<4} | cv {s.mean():.3f}")

> **The rule:** a smaller `learning_rate` needs more `max_iter`. Small steps, many of them,
> generalise better than few large steps - but cost more time. Unlike a Random Forest,
> **too many boosting rounds does overfit**, so use `early_stopping=True`.

## 4. Head-to-head

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

contenders = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
}

rows = []
for name, m in contenders.items():
    s = cross_val_score(m, X, y, cv=5, scoring="roc_auc")
    rows.append({"model": name, "roc_auc": s.mean().round(3), "std": s.std().round(3)})

pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)

On a small, clean dataset like this one, logistic regression is often competitive.
**Always benchmark against a simple model.** If a boosted ensemble only beats logistic
regression by 0.005 AUC, ship the logistic regression - it is faster, interpretable and
easier to debug.

## 5. Feature importance, and its trap

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values()
importances.plot.barh(figsize=(8, 6), color="#4C78A8")
plt.title("Built-in (impurity) feature importance")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

**The trap:** built-in importance is biased towards high-cardinality features - a column
with many distinct values gets more chances to look useful. It is computed on the
*training* data, so it also rewards features the model overfitted on.

**Permutation importance** is the honest version: shuffle one column in the *test* set and
measure how much the score drops. If nothing drops, the feature was not being used.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42, scoring="roc_auc")

comparison = pd.DataFrame({
    "built-in": rf.feature_importances_,
    "permutation": perm.importances_mean,
}, index=X.columns).sort_values("permutation", ascending=False).round(4)
comparison

Where the two columns disagree, trust permutation importance. And remember: importance
means *the model used it*, never *it causes the outcome*.

---
## Exercises

1. Plot a `max_depth=2` tree and write out, in plain sentences, the rules it learned.
   Would you be comfortable showing them to a domain expert?
2. Set `max_features=1` on the Random Forest. Score drops - explain why, in terms of what
   each individual tree can see.
3. Turn on `early_stopping=True` for `HistGradientBoostingClassifier` and print
   `.n_iter_`. How many rounds did it actually need?
4. Run the head-to-head comparison on the imbalanced dataset from notebook 05, scoring by
   `average_precision`. Does the ranking change?
5. **Stretch:** `pip install xgboost lightgbm` and add `XGBClassifier` and
   `LGBMClassifier` to the comparison. These are what Kaggle competitions actually run on.